# Benchmark audité du corpus GeNIS 2025 — pipeline expérimental complet

**Article :** *A Leakage-Audited Benchmark of Deep and Ensemble Detectors on the GeNIS 2025 Corpus*
— Ala Bahri, Farah Jemili, Mohamed Mosbah.

Ce notebook exécute **l'intégralité de l'étude** et produit **tous les livrables** :
modèles entraînés sauvegardés, figures 300 dpi (article + rapport de thèse),
tables LaTeX et fichier de résultats.

---

## Sommaire

| § | Étape | Livrables |
|---|-------|-----------|
| 1 | Configuration, données, reprise | — |
| 2 | Exploration des données (EDA) | Fig. 1, figures annexes A1–A5 |
| 3 | Prétraitement : features, splits, sondes de raccourci | Table 1, Fig. A6 |
| 4 | Définition des 12 modèles | Table 2 |
| 5 | Entraînement : défaut → audit → réglage → bras réglé → autoencodeur | Table 3–5 |
| 6 | Évaluation : tables, figures, calibration, coût, statistiques | Fig. 2–8, A7–A15 |
| 7 | **Sauvegarde des modèles** et export | `article1_models.zip` |

## Protocole (gelé)

- **Protocole propre** : scaler ajusté sur le train seul, distribution naturelle, aucune rééquilibration du test.
- **Trois conditions de features** : `full` (avec colonnes positionnelles) → `clean` (sans) → `audited` (après liste noire issue de l'audit §5.2).
- **Deux splits** : stratifié 60/20/20 × 5 graines, et **temporel par classe** (le test de chaque classe est postérieur à son train). Le split chronologique *global* est dégénéré sur GeNIS — documenté comme résultat (§3.5), pas utilisé comme protocole.
- **Deux bras d'hyperparamètres** : `défaut` (architectures BAg-IDS publiées + valeurs usuelles) et `réglé` (recherche aléatoire à budget déclaré, sélection sur validation).

## Exécution

- Runtime **GPU** (T4 suffit). Prérequis : `2-flows.zip` dans `MyDrive/GeNIS/`.
- *Exécution → Tout exécuter*. **Durée totale ≈ 8–11 h**, mais chaque run est sauvegardé
  immédiatement : en cas de déconnexion, relancer « Tout exécuter » reprend exactement
  où le notebook s'était arrêté. Comptez **2 à 3 sessions**.
- Sorties persistantes : `MyDrive/GeNIS/article1_final/`.

| Bloc | Durée indicative (T4) |
|---|---|
| §2–§3 EDA, splits, sondes | 15–25 min |
| §5.1 bras défaut, conditions `full`/`clean` | 1 h |
| §5.2 audit → liste noire | 15 min |
| §5.3 bras défaut, condition `audited` (5 graines + temporel) | 2–3 h |
| §5.4 recherche d'hyperparamètres | 1,5–2 h |
| §5.5 bras réglé | 2–3 h |
| §5.6 autoencodeur | 20 min |
| §6.4 multi-intervalles | 1–1,5 h |
| §6.5 banc de coût, §6.6 statistiques, §6.7 figures | 30 min |
| §7 sauvegarde et export | 5 min |

**À renvoyer à la fin :** `article1_results.json`, `article1_figures.zip`, et (facultatif) `article1_models.zip`.


## 1. Configuration

In [ ]:
# 1.1 — Environnement, dependances, constantes du protocole
import os, sys, glob, json, time, math, shutil, pathlib, platform, itertools, warnings, pickle, io
import numpy as np, pandas as pd, sklearn, scipy, joblib
import tensorflow as tf
import matplotlib, matplotlib.pyplot as plt
from IPython.display import display
try:
    import xgboost, lightgbm
except ImportError:
    %pip -q install xgboost lightgbm
    import xgboost, lightgbm

warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
                     "axes.grid": True, "grid.alpha": .3, "axes.axisbelow": True,
                     "figure.facecolor": "white", "savefig.bbox": "tight"})

# ---------------- constantes du protocole (gelees) ----------------
SEEDS         = [1, 2, 3, 4, 5]      # graines des splits stratifies
SPLIT_FRAC    = (0.60, 0.20, 0.20)
WINDOW_GAP_S  = 1800                 # 30 min sans flux => nouvelle fenetre d'activite
BOOTSTRAP_B   = 1000
ECE_BINS      = 15

# ---------------- interrupteurs de cout ----------------
RUN_HPO       = True    # §5.4-5.5 recherche d'hyperparametres + bras regle  (~3-5 h)
RUN_INTERVALS = True    # §6.4 etude 5/10/30 s                                (~1-1.5 h)
RUN_COST      = True    # §6.5 banc de cout CPU                               (~15 min)
SAVE_MODELS   = True    # §7 sauvegarde des modeles de reference

GPU = bool(tf.config.list_physical_devices("GPU"))
print(f"python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__}")
print(f"sklearn {sklearn.__version__} | xgboost {xgboost.__version__} | lightgbm {lightgbm.__version__}")
print(f"tensorflow {tf.__version__}")
print("GPU :", "oui" if GPU else "NON — les sections profondes seront tres lentes")


In [ ]:
# 1.2 — Drive, telechargement du corpus, reprise automatique
from google.colab import drive
drive.mount('/content/drive')

WORK = pathlib.Path("/content/genis"); WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

SAVE   = pathlib.Path("/content/drive/MyDrive/GeNIS/article1_final")   # dossier NEUF
PROBS  = SAVE / "probs"      # probabilites par run (float16)
FIGS   = SAVE / "figures"    # figures article
SUPP   = SAVE / "figures_annexe"  # figures rapport de these
TABS   = SAVE / "tables"     # fragments LaTeX
MODELS = SAVE / "models"     # modeles entraines
for p in (SAVE, PROBS, FIGS, SUPP, TABS, MODELS): p.mkdir(parents=True, exist_ok=True)

RES_PATH = SAVE / "article1_results.json"
RESULTS  = json.loads(RES_PATH.read_text()) if RES_PATH.exists() else {}
for k in ("models", "meta", "history"): RESULTS.setdefault(k, {})
RESULTS["meta"].update({
    "updated": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "protocol": {"seeds": SEEDS, "split": SPLIT_FRAC, "ece_bins": ECE_BINS,
                 "bootstrap": BOOTSTRAP_B, "window_gap_s": WINDOW_GAP_S},
    "env": {"python": platform.python_version(), "tensorflow": tf.__version__,
            "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
            "lightgbm": lightgbm.__version__, "gpu": GPU}})
def save_results():
    RES_PATH.write_text(json.dumps(RESULTS, indent=1, default=float), encoding="utf-8")
save_results()
print(f"dossier de sortie : {SAVE}")
print(f"runs deja calcules et reutilises : {len(RESULTS['models'])}")

drive_zip = "/content/drive/MyDrive/GeNIS/2-flows.zip"
if not pathlib.Path("flows").exists():
    if pathlib.Path(drive_zip).exists():
        print("copie de 2-flows.zip depuis Drive…"); shutil.copy(drive_zip, "2-flows.zip")
    else:
        print("telechargement depuis Zenodo (~380 Mo)…")
        !wget -q --show-progress "https://zenodo.org/records/14919237/files/2-flows.zip?download=1" -O 2-flows.zip
    !unzip -o -q 2-flows.zip -d flows
csvs = sorted(glob.glob("flows/**/*.csv", recursive=True))
assert csvs, "aucun CSV trouve — verifier l'archive"
print(f"{len(csvs)} fichiers CSV disponibles")


## 2. Exploration des données (EDA)

GeNIS fournit **le même trafic** agrégé à quatre intervalles (5/10/30/60 s) : quatre vues
d'un même corpus, à ne jamais mélanger. Cette section établit les volumes, le déséquilibre,
le schéma des features (exporteur HERA/Argus) et la **structure temporelle** de la capture,
qui conditionne tout le protocole.


In [ ]:
# 2.1 — Volumes et desequilibre par intervalle  (Table 1 de l'article)
INTERVALS = ["5", "10", "30", "60"]
if "interval_stats" not in RESULTS:
    st = {}
    for iv in INTERVALS:
        counts = {pathlib.Path(f).stem: sum(1 for _ in open(f, "rb")) - 1
                  for f in csvs if f"flows-{iv}-sec" in f}
        tot = sum(counts.values()); ben = sum(v for k, v in counts.items() if k.startswith("benign"))
        st[iv] = {"files": counts, "total": tot, "benign": ben, "benign_share": ben / tot}
    RESULTS["interval_stats"] = st; save_results()

t1 = pd.DataFrame({iv: {"flux total": RESULTS["interval_stats"][iv]["total"],
                        "flux benins": RESULTS["interval_stats"][iv]["benign"],
                        "part benin (%)": round(RESULTS["interval_stats"][iv]["benign_share"] * 100, 2)}
                   for iv in INTERVALS}).T
t1.index.name = "intervalle (s)"; display(t1)

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(INTERVALS, [RESULTS["interval_stats"][iv]["total"] for iv in INTERVALS], color="#4C72B0")
ax[0].set_title("Nombre de flux"); ax[0].set_xlabel("intervalle d'agregation (s)")
ax[1].bar(INTERVALS, [RESULTS["interval_stats"][iv]["benign_share"] * 100 for iv in INTERVALS],
          color="#55A868")
ax[1].set_title("Part de trafic benin (%)"); ax[1].set_xlabel("intervalle d'agregation (s)")
plt.savefig(SUPP / "A1_intervals.png"); plt.savefig(SUPP / "A1_intervals.pdf"); plt.show()
# Lecture : la part de benin varie avec l'intervalle. L'accuracy brute n'est donc comparable
# ni entre intervalles ni avec d'autres corpus -> macro-F1 et MCC dans tout le papier.


In [ ]:
# 2.2 — Tranche 60 s : chargement et unification des labels (11 -> 9 classes)
def load_slice(iv):
    files = [c for c in csvs if f"flows-{iv}-sec" in c]
    d = pd.concat([pd.read_csv(c, low_memory=False) for c in files], ignore_index=True)
    ysub = d["SubCategoryLabel"].astype(str).str.strip()
    y9 = ysub.where(~ysub.str.startswith("benign"), "benign")   # fusion des 3 profils benins
    t = pd.to_numeric(d["StartTime"], errors="coerce")
    assert t.notna().all(), "StartTime non numerique"
    return d, y9, t

df, y9_raw, t_start = load_slice("60")
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder(); y = le.fit_transform(y9_raw)
CLASS_NAMES = list(le.classes_); C = len(CLASS_NAMES)
BENIGN_IDX  = CLASS_NAMES.index("benign")
t_np = t_start.values.astype(np.float64)

print(f"tranche 60 s : {len(df):,} flux | {df.shape[1]} colonnes brutes | {C} classes")
dist = pd.DataFrame({"flux": pd.Series(y9_raw).value_counts(),
                     "part (%)": (pd.Series(y9_raw).value_counts(normalize=True) * 100).round(2)})
display(dist)
print(f"ratio de desequilibre max/min : {dist['flux'].max() / dist['flux'].min():.1f}:1")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
cols = ["#55A868" if c == "benign" else "#C44E52" for c in dist.index]
dist["flux"].plot.bar(ax=ax[0], color=cols); ax[0].set_yscale("log")
ax[0].set_ylabel("flux (echelle log)"); ax[0].set_title("Distribution naturelle des 9 classes")
ax[1].pie([len(y) - (y == BENIGN_IDX).sum(), (y == BENIGN_IDX).sum()],
          labels=["attaque", "benin"], autopct="%1.1f%%", colors=["#C44E52", "#55A868"])
ax[1].set_title("Repartition binaire")
plt.savefig(SUPP / "A2_class_distribution.png"); plt.savefig(SUPP / "A2_class_distribution.pdf"); plt.show()


In [ ]:
# 2.3 — Structure temporelle : fenetres d'activite et chronologie (Figure 1)
def activity_windows(times, gap=WINDOW_GAP_S):
    ts = np.sort(times); brk = np.where(np.diff(ts) > gap)[0]
    return [(s[0], s[-1], len(s)) for s in np.split(ts, brk + 1)]

t0 = t_np.min()
rows = []
for i, cn in enumerate(CLASS_NAMES):
    w = activity_windows(t_np[y == i])
    rows.append({"classe": cn, "flux": int((y == i).sum()), "fenetres": len(w),
                 "duree active (h)": round(sum(e - s for s, e, _ in w) / 3600, 2),
                 "debut (h)": round((w[0][0] - t0) / 3600, 1),
                 "fin (h)": round((w[-1][1] - t0) / 3600, 1)})
twin = pd.DataFrame(rows).set_index("classe"); display(twin)
RESULTS["activity_windows"] = twin.reset_index().to_dict("records")
RESULTS["slice60_span_h"] = float((t_np.max() - t_np.min()) / 3600)

fig, ax = plt.subplots(figsize=(9, 3.4))
rng = np.random.RandomState(0)
for i, cn in enumerate(CLASS_NAMES):
    ix = np.where(y == i)[0]; ix = rng.choice(ix, size=min(3000, len(ix)), replace=False)
    ax.plot((t_np[ix] - t0) / 3600, np.full(len(ix), i), "|", ms=5,
            color="#55A868" if i == BENIGN_IDX else "#C44E52", alpha=.3)
ax.set_yticks(range(C)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("heures depuis le debut de la capture (6-12 fevrier 2025)")
ax.set_title("Figure 1 — fenetres temporelles par classe : etroites et disjointes")
plt.savefig(FIGS / "fig1_timeline.png"); plt.savefig(FIGS / "fig1_timeline.pdf"); plt.show()
save_results()
# Lecture : le benin (vert) precede toutes les attaques (calendrier de capture), et chaque
# famille occupe des fenetres etroites. La POSITION temporelle d'un flux predit donc presque
# sa classe (verifie en §3.4) et un split chronologique global est degenere (§3.5).


## 3. Prétraitement

**Catégorisation explicite des colonnes** — listes nominatives, jamais de motifs de
chaînes (source classique de deux erreurs : garder `StartTime` comme feature, ou
supprimer par accident `SIntPktIdl` avec un filtre `"id"`) :

| Catégorie | Exemples | Traitement |
|---|---|---|
| **Identifiants** | `SrcAddr`, `Dport`, `FlowID`, `SrcMac` | exclus de **toutes** les conditions |
| **Positionnelles** | `StartTime`, `LastTime`, `Rank`, `Seq` | condition `full` uniquement |
| **Comportementales** | `Dur`, `TotPkts`, `sTtl`, `SIntPktIdl` | candidates ; l'audit §5.2 tranche |
| **Labels** | `BinaryLabel`, `CategoryLabel`, `SubCategoryLabel` | jamais features |


In [ ]:
# 3.1 — Jeux de features et diagnostic de redondance
IDENT_LIST = ["FlowID", "AutoId", "SrcAddr", "DstAddr", "Ssaddr", "Sdaddr",
              "SrcMac", "DstMac", "SrcOui", "DstOui", "Sport", "Dport",
              "sIpId", "dIpId", "sMpls", "dMpls", "sAS", "dAS", "iAS",
              "sCo", "dCo", "sVid", "dVid"]
POS_LIST = ["StartTime", "LastTime", "Rank", "Seq"]
LAB_LIST = ["BinaryLabel", "CategoryLabel", "SubCategoryLabel"]

def feature_sets(d):
    ident = [c for c in IDENT_LIST if c in d.columns]
    pos   = [c for c in POS_LIST if c in d.columns]
    num = (d.drop(columns=ident + LAB_LIST, errors="ignore")
             .select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan))
    nu = num.nunique(dropna=True); const = nu[nu <= 1].index.tolist()
    num = num.drop(columns=const).fillna(0.0).astype(np.float32)
    return num, list(num.columns), [c for c in num.columns if c not in pos], pos, ident, const

X_all, F_FULL, F_CLEAN, POSITIONAL, IDENTIFIERS, DROPPED_CONST = feature_sets(df)
print(f"features 'full'  : {len(F_FULL)}  (avec positionnelles {POSITIONAL})")
print(f"features 'clean' : {len(F_CLEAN)}")
print(f"identifiants exclus partout : {len(IDENTIFIERS)} -> {IDENTIFIERS}")
print(f"colonnes constantes ecartees : {len(DROPPED_CONST)} -> {DROPPED_CONST}")

sub = X_all[F_CLEAN].sample(min(20000, len(X_all)), random_state=0)
corr = np.corrcoef(sub.values, rowvar=False)
hi = int((np.abs(np.triu(corr, 1)) > 0.95).sum())
print(f"paires de features |r| > 0.95 : {hi}")
RESULTS["slice60"] = {"n": int(len(y)), "classes": CLASS_NAMES,
                      "class_counts": pd.Series(y9_raw).value_counts().to_dict(),
                      "benign_share": float((y == BENIGN_IDX).mean()),
                      "features_full": F_FULL, "features_clean": F_CLEAN,
                      "positional": POSITIONAL, "identifiers_excluded": IDENTIFIERS,
                      "constant_dropped": DROPPED_CONST, "high_corr_pairs": hi}

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title(f"Correlation des {len(F_CLEAN)} features comportementales")
ax.set_xticks([]); ax.set_yticks([]); plt.colorbar(im, label="coefficient de Pearson")
plt.savefig(SUPP / "A3_feature_correlation.png"); plt.savefig(SUPP / "A3_feature_correlation.pdf"); plt.show()
save_results()


In [ ]:
# 3.2 — Projection PCA : separabilite visuelle des classes (figure d'annexe)
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler

rng = np.random.RandomState(0)
ix = np.concatenate([rng.choice(np.where(y == c)[0], size=min(1500, (y == c).sum()), replace=False)
                     for c in range(C)])
Xs = RobustScaler().fit_transform(X_all[F_CLEAN].values[ix])
Xs = np.nan_to_num(Xs, nan=0., posinf=0., neginf=0.)
pc = PCA(n_components=2, random_state=0).fit(Xs)
Z = pc.transform(Xs)

fig, ax = plt.subplots(figsize=(6.5, 5))
cmap = plt.get_cmap("tab10")
for c in range(C):
    m = y[ix] == c
    ax.scatter(Z[m, 0], Z[m, 1], s=4, alpha=.45, label=CLASS_NAMES[c],
               color="#55A868" if c == BENIGN_IDX else cmap(c % 10))
ax.set_xlabel(f"CP1 ({pc.explained_variance_ratio_[0]*100:.1f} % de variance)")
ax.set_ylabel(f"CP2 ({pc.explained_variance_ratio_[1]*100:.1f} % de variance)")
ax.set_title("Projection PCA des features comportementales (echantillon equilibre)")
ax.legend(fontsize=7, markerscale=2, ncol=2)
plt.savefig(SUPP / "A4_pca.png"); plt.savefig(SUPP / "A4_pca.pdf"); plt.show()
RESULTS["pca_explained_2c"] = [float(v) for v in pc.explained_variance_ratio_[:2]]
save_results()


In [ ]:
# 3.3 — Splits geles : stratifie x5 + temporel PAR CLASSE
from sklearn.model_selection import train_test_split

def temporal_split_per_class(y_, t_, frac=SPLIT_FRAC):
    # Pour chaque classe : tri temporel puis 60/20/20 => tout flux de test est
    # posterieur a tous les flux d'entrainement de SA classe, et aucune classe ne disparait.
    tr, va, te = [], [], []
    for c in range(C):
        ix = np.where(y_ == c)[0]; ix = ix[np.argsort(t_[ix], kind="stable")]
        n = len(ix); a, b = int(frac[0] * n), int((frac[0] + frac[1]) * n)
        tr.append(ix[:a]); va.append(ix[a:b]); te.append(ix[b:])
    return np.concatenate(tr), np.concatenate(va), np.concatenate(te)

SPLIT_KEYS  = [f"strat_seed{s}" for s in SEEDS] + ["temporal"]
SPLITS_PATH = SAVE / "frozen_splits_60s.npz"
if SPLITS_PATH.exists():
    z = np.load(SPLITS_PATH)
    splits = {k: (z[f"{k}_train"], z[f"{k}_val"], z[f"{k}_test"]) for k in SPLIT_KEYS}
    print("splits recharges depuis le gel")
else:
    idx = np.arange(len(y)); splits = {}
    for s in SEEDS:
        itr, itmp = train_test_split(idx, test_size=0.40, random_state=s, stratify=y)
        iva, ite  = train_test_split(itmp, test_size=0.50, random_state=s, stratify=y[itmp])
        splits[f"strat_seed{s}"] = (itr, iva, ite)
    splits["temporal"] = temporal_split_per_class(y, t_np)
    np.savez_compressed(SPLITS_PATH, **{f"{k}_{p}": v for k, (a, b, c_) in splits.items()
                                        for p, v in zip(("train", "val", "test"), (a, b, c_))})
    print(f"splits generes et geles -> {SPLITS_PATH}")

tr, va, te = splits["temporal"]
chk = pd.DataFrame({p: pd.Series(y[i_]).value_counts().reindex(range(C), fill_value=0).values
                    for p, i_ in zip(("train", "val", "test"), (tr, va, te))}, index=CLASS_NAMES)
display(chk)
ok = all(t_np[tr][y[tr] == c].max() <= t_np[te][y[te] == c].min()
         for c in range(C) if (y[tr] == c).any() and (y[te] == c).any())
print("invariant temporel (test posterieur au train, par classe) :", "OK" if ok else "VIOLE")
print("toutes les classes presentes dans les 3 partitions :", bool((chk > 0).all().all()))
RESULTS["temporal_class_table"] = chk.to_dict(); save_results()


In [ ]:
# 3.4 — Sondes de raccourci : que predit la seule POSITION temporelle ?
from sklearn.tree import DecisionTreeClassifier

def probe(cols, key):
    tr_, _, te_ = splits[key]
    Xp = df[cols].astype(np.float64).values
    clf = DecisionTreeClassifier(random_state=1).fit(Xp[tr_], y[tr_])
    return float((clf.predict(Xp[te_]) == y[te_]).mean())

if "shortcut_probes" not in RESULTS:
    pr = {"chance_majority": float(pd.Series(y).value_counts(normalize=True).max())}
    for nm, cols in [("starttime_only", ["StartTime"]), ("positional_all", POSITIONAL)]:
        accs = [probe(cols, f"strat_seed{s}") for s in SEEDS]
        pr[nm] = {"strat_mean": float(np.mean(accs)), "strat_std": float(np.std(accs)),
                  "temporal": probe(cols, "temporal")}
    RESULTS["shortcut_probes"] = pr; save_results()

pr = RESULTS["shortcut_probes"]
print(f"hasard (classe majoritaire)  : {pr['chance_majority']:.4f}")
for nm, lab in [("starttime_only", "StartTime seul"), ("positional_all", "positionnelles")]:
    print(f"{lab:16s} : stratifie {pr[nm]['strat_mean']:.4f} (+/- {pr[nm]['strat_std']:.4f})"
          f" | temporel {pr[nm]['temporal']:.4f}")

fig, ax = plt.subplots(figsize=(6, 3.2))
labs = ["StartTime seul", "positionnelles"]
xs = np.arange(len(labs)); w = .35
ax.bar(xs - w/2, [pr[k]["strat_mean"] for k in ("starttime_only", "positional_all")], w,
       yerr=[pr[k]["strat_std"] for k in ("starttime_only", "positional_all")],
       label="split stratifie", color="#C44E52", capsize=3)
ax.bar(xs + w/2, [pr[k]["temporal"] for k in ("starttime_only", "positional_all")], w,
       label="split temporel", color="#4C72B0")
ax.axhline(pr["chance_majority"], ls="--", c="k", lw=.8, label="hasard (classe majoritaire)")
ax.set_xticks(xs); ax.set_xticklabels(labs); ax.set_ylabel("accuracy 9 classes")
ax.set_title("Sondes de raccourci temporel"); ax.legend(fontsize=8)
plt.savefig(SUPP / "A5_shortcut_probes.png"); plt.savefig(SUPP / "A5_shortcut_probes.pdf"); plt.show()
# Lecture : un ecart massif entre les deux barres signifie que le split stratifie recompense
# la MEMORISATION DU CALENDRIER de capture. C'est le chiffre d'accroche de l'introduction (RQ2).


In [ ]:
# 3.5 — Pourquoi PAS un split chronologique GLOBAL (resultat, pas protocole)
o = np.argsort(t_np, kind="stable"); n = len(o)
g_tr, g_te = o[:int(.6 * n)], o[int(.8 * n):]
gtab = pd.DataFrame({"train (60 % premiers)": pd.Series(y[g_tr]).value_counts().reindex(range(C), fill_value=0).values,
                     "test (20 % derniers)":  pd.Series(y[g_te]).value_counts().reindex(range(C), fill_value=0).values},
                    index=CLASS_NAMES)
display(gtab)
missing = [CLASS_NAMES[c] for c in range(C) if gtab.iloc[c, 1] > 0 and gtab.iloc[c, 0] == 0]
print(f"classes du test jamais vues a l'entrainement : {missing or 'aucune'}")
RESULTS["global_chrono_table"] = gtab.to_dict(); save_results()
# Lecture : le decoupage chronologique global place dans le test des blocs entiers de classes
# absentes du train (consequence directe du calendrier : benin les 6-8 fevrier, attaques les
# 10-12). Tout modele y obtient une accuracy proche de zero : il mesure le CALENDRIER, pas la
# generalisation. D'ou le split temporel PAR CLASSE de §3.3.


## 4. Modèles

Douze modèles supervisés et un détecteur d'anomalie, tous sous le même protocole.
Le trio RNN/CNN/DNN reprend **exactement** les architectures publiées dans BAg-IDS
(provenance des détecteurs), et le bras « réglé » (§5.4) explore autour sans changer
la famille.

| Modèle | Famille | Rôle | Configuration par défaut |
|---|---|---|---|
| `majority` | baseline triviale | plancher de référence | — |
| `logreg` | linéaire | baseline calibrable | `max_iter=1000` |
| `nb` | bayésien naïf | baseline probabiliste | `GaussianNB` |
| `knn` | à mémoire | baseline instance | `k=5` |
| `rf` | forêt aléatoire | référence « bagging » | 200 arbres |
| `xgboost` | boosting | état de l'art tabulaire | 300 arbres, prof. 8, lr 0,1 |
| `lightgbm` | boosting | état de l'art tabulaire | 300 arbres, 63 feuilles, lr 0,1 |
| `rnn` | récurrent | **provenance BAg-IDS** | SimpleRNN(64) → Dense(64) |
| `cnn` | convolutif 1D | **provenance BAg-IDS** | Conv(64,3) → Conv(32,3) → Dense(64) |
| `dnn` | dense | **provenance BAg-IDS** | 128 → 64 → softmax |
| `ftt` | FT-Transformer | profond tabulaire moderne | d=64, 3 blocs, 8 têtes |
| `ae` | autoencodeur | anomalie (bénin seul), **évalué à part** en AUROC | 64-32-16-32-64 |

**Exclusions déclarées** : le SVM à noyau (entraînement en O(n²) : inentraînable sur
~200 000 flux à budget égal ; la régression logistique couvre la famille linéaire) et le
trio k-means/EM/SOM (remplacé par l'autoencodeur, représentant moderne de la détection
d'anomalies).

Entraînement profond : Adam, 30 époques max, early stopping (patience 5), `class_weight`
équilibré — le rééquilibrage agit **à l'entraînement seulement**, le test reste à sa
distribution naturelle.


In [ ]:
# 4.1 — Definitions parametrees (les valeurs par defaut = bras "defaut")
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                             roc_auc_score, confusion_matrix, roc_curve,
                             precision_recall_curve, classification_report)

DEFAULTS_SK = {"majority": {}, "logreg": {"max_iter": 1000}, "nb": {},
               "knn": {"n_neighbors": 5},
               "rf": {"n_estimators": 200},
               "xgboost": {"n_estimators": 300, "max_depth": 8, "learning_rate": 0.1},
               "lightgbm": {"n_estimators": 300, "num_leaves": 63, "learning_rate": 0.1}}
def make_sk(name, params=None):
    p = {**DEFAULTS_SK[name], **(params or {})}
    if name == "majority": return DummyClassifier(strategy="most_frequent")
    if name == "logreg":   return LogisticRegression(n_jobs=-1, **p)
    if name == "nb":       return GaussianNB(**p)
    if name == "knn":      return KNeighborsClassifier(n_jobs=-1, **p)
    if name == "rf":       return RandomForestClassifier(n_jobs=-1, random_state=0, **p)
    if name == "xgboost":  return XGBClassifier(tree_method="hist", n_jobs=-1, random_state=0,
                                                eval_metric="mlogloss", **p)
    if name == "lightgbm": return LGBMClassifier(n_jobs=-1, random_state=0, verbose=-1, **p)
    raise KeyError(name)
FAST = list(DEFAULTS_SK.keys())

# Architectures BAg-IDS telles que publiees (bras "defaut" = provenance).
DEFAULTS_DEEP = {
    "dnn": {"h1": 128, "h2": 64, "d1": .3, "d2": .2, "lr": 1e-3, "bs": 256},
    "cnn": {"f1": 64, "f2": 32, "dense": 64, "drop": .3, "lr": 1e-3, "bs": 256},
    "rnn": {"units": 64, "dense": 64, "drop": .3, "lr": 1e-3, "bs": 256},
    "ftt": {"d": 64, "heads": 8, "blocks": 3, "ff": 128, "drop": .1, "lr": 5e-4, "bs": 512}}

def build_dnn(F, p=None):
    q = {**DEFAULTS_DEEP["dnn"], **(p or {})}
    return models.Sequential([layers.Input((F,)),
        layers.Dense(q["h1"], activation="relu"), layers.Dropout(q["d1"]),
        layers.Dense(q["h2"], activation="relu"), layers.Dropout(q["d2"]),
        layers.Dense(C, activation="softmax")], name="dnn")
def build_cnn(F, p=None):
    q = {**DEFAULTS_DEEP["cnn"], **(p or {})}
    return models.Sequential([layers.Input((F, 1)),
        layers.Conv1D(q["f1"], 3, activation="relu", padding="same"), layers.MaxPooling1D(2),
        layers.Conv1D(q["f2"], 3, activation="relu", padding="same"), layers.Flatten(),
        layers.Dense(q["dense"], activation="relu"), layers.Dropout(q["drop"]),
        layers.Dense(C, activation="softmax")], name="cnn")
def build_rnn(F, p=None):
    q = {**DEFAULTS_DEEP["rnn"], **(p or {})}
    return models.Sequential([layers.Input((F, 1)),
        layers.SimpleRNN(q["units"], activation="relu"), layers.Dropout(q["drop"]),
        layers.Dense(q["dense"], activation="relu"),
        layers.Dense(C, activation="softmax")], name="rnn")

class FeatureTokenizer(layers.Layer):           # FT-Transformer (Gorishniy et al., 2021)
    def __init__(self, d, **kw): super().__init__(**kw); self.d = d
    def build(self, shape):
        F = int(shape[-1])
        self.w = self.add_weight(shape=(F, self.d), initializer="glorot_uniform", name="w")
        self.b = self.add_weight(shape=(F, self.d), initializer="zeros", name="b")
    def call(self, x): return x[:, :, None] * self.w + self.b
    def get_config(self): return {**super().get_config(), "d": self.d}
class ClsToken(layers.Layer):
    def __init__(self, d, **kw): super().__init__(**kw); self.d = d
    def build(self, shape):
        self.cls = self.add_weight(shape=(1, 1, self.d), initializer="glorot_uniform", name="cls")
    def call(self, x): return tf.concat([tf.tile(self.cls, [tf.shape(x)[0], 1, 1]), x], axis=1)
    def get_config(self): return {**super().get_config(), "d": self.d}
def build_ftt(F, p=None):
    q = {**DEFAULTS_DEEP["ftt"], **(p or {})}
    d, heads, blocks, ff, drop = q["d"], q["heads"], q["blocks"], q["ff"], q["drop"]
    inp = layers.Input((F,)); x = ClsToken(d)(FeatureTokenizer(d)(inp))
    for _ in range(blocks):
        h = layers.LayerNormalization()(x)
        h = layers.MultiHeadAttention(num_heads=heads, key_dim=max(1, d // heads), dropout=drop)(h, h)
        x = layers.Add()([x, h])
        h = layers.LayerNormalization()(x)
        h = layers.Dense(ff, activation="gelu")(h); h = layers.Dropout(drop)(h)
        x = layers.Add()([x, layers.Dense(d)(h)])
    return models.Model(inp, layers.Dense(C, activation="softmax")(
        layers.LayerNormalization()(x[:, 0])), name="ftt")

DEEP = {"rnn": build_rnn, "cnn": build_cnn, "dnn": build_dnn, "ftt": build_ftt}
CUSTOM_OBJECTS = {"FeatureTokenizer": FeatureTokenizer, "ClsToken": ClsToken}
def shape_for(name, A): return A if name in ("dnn", "ftt") else A.reshape(-1, A.shape[1], 1)
def build_ae(F):
    return models.Sequential([layers.Input((F,)),
        layers.Dense(64, activation="relu"), layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"), layers.Dense(32, activation="relu"),
        layers.Dense(64, activation="relu"), layers.Dense(F)], name="ae")

ALL_MODELS = FAST + list(DEEP)
print("modeles :", ALL_MODELS, "+ autoencodeur")


In [ ]:
# 4.2 — Boucles d'entrainement, evaluation, persistance
def make_xy(cols, key, return_scaler=False):
    tr_, va_, te_ = splits[key]
    X = X_all[cols].values
    sc = RobustScaler().fit(X[tr_])                       # scaler ajuste sur le TRAIN seul
    parts = [np.nan_to_num(sc.transform(X[i_]), nan=0., posinf=0., neginf=0.).astype(np.float32)
             for i_ in (tr_, va_, te_)]
    out = (parts, (y[tr_], y[va_], y[te_]))
    return (*out, sc) if return_scaler else out

def evaluate(y_true, probs, fit_t, pred_t):
    pred = probs.argmax(1); present = np.unique(y_true)
    is_att, pred_att = y_true != BENIGN_IDX, pred != BENIGN_IDX
    p_att = 1.0 - probs[:, BENIGN_IDX]
    return {"accuracy": float((pred == y_true).mean()),
            "macro_f1": float(f1_score(y_true, pred, labels=present, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(y_true, pred, average="weighted", zero_division=0)),
            "mcc": float(matthews_corrcoef(y_true, pred)),
            "per_class_f1": {CLASS_NAMES[i]: float(v) for i, v in zip(range(C),
                f1_score(y_true, pred, labels=range(C), average=None, zero_division=0))},
            "binary": {"detection_f1": float(f1_score(is_att, pred_att, zero_division=0)),
                       "fpr": float(pred_att[~is_att].mean()) if (~is_att).any() else None,
                       "fnr": float((~pred_att[is_att]).mean()) if is_att.any() else None,
                       "pr_auc": float(average_precision_score(is_att, p_att)) if 0 < is_att.mean() < 1 else None,
                       "roc_auc": float(roc_auc_score(is_att, p_att)) if 0 < is_att.mean() < 1 else None},
            "fit_time_s": round(fit_t, 2), "predict_time_s": round(pred_t, 3),
            "n_test": int(len(y_true))}

def kfile(key): return PROBS / f"{key.replace('|', '_').replace('#', '-')}.npz"
def done(key):  return key in RESULTS["models"] and kfile(key).exists()
def load_probs(key):
    z = np.load(kfile(key)); return z["probs_val"].astype(np.float64), z["probs_test"].astype(np.float64)

def record(key, y_true, pva, pte, fit_t, pred_t):
    RESULTS["models"][key] = evaluate(y_true, pte, fit_t, pred_t)
    np.savez_compressed(kfile(key), probs_val=pva.astype(np.float16), probs_test=pte.astype(np.float16))
    r = RESULTS["models"][key]; fpr = r["binary"]["fpr"]
    print(f"  {key:44s} acc {r['accuracy']:.4f}  mF1 {r['macro_f1']:.4f}  MCC {r['mcc']:.4f}"
          f"  FPR {(f'{fpr:.4%}' if fpr is not None else 'n/a'):>9s}  [{fit_t:.0f}s]")
    save_results()

# --- persistance : on garde les modeles de reference (condition auditee, graine 1) ---
REF_SPLIT, REF_TAG = "strat_seed1", "audited"
def persist_sk(model, mname, tag, split, arm):
    if SAVE_MODELS and split == REF_SPLIT and tag in (REF_TAG, "clean"):
        joblib.dump(model, MODELS / f"{mname}_{arm}_{tag}_{split}.joblib", compress=3)
def persist_keras(m, mname, tag, split, arm):
    if SAVE_MODELS and split == REF_SPLIT and tag in (REF_TAG, "clean"):
        m.save(MODELS / f"{mname}_{arm}_{tag}_{split}.keras")

def fit_sk(mname, Xtr, ytr, params=None):
    present = np.unique(ytr); model = make_sk(mname, params)
    model.fit(Xtr, np.searchsorted(present, ytr) if len(present) < C else ytr)
    def predict_full(X):
        p = model.predict_proba(X)
        if len(present) == C and list(getattr(model, "classes_", range(C))) == list(range(C)):
            return p
        out = np.zeros((len(X), C), dtype=np.float32)
        out[:, (present if len(present) < C else np.asarray(model.classes_, int))] = p
        return out
    return model, predict_full

def train_fast(mname, cols, key_split, tag, params=None, arm="default"):
    key = f"{mname}{'' if arm == 'default' else '#tuned'}|{tag}|{key_split}"
    if done(key): return
    (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(cols, key_split)
    t0 = time.time(); model, pf = fit_sk(mname, Xtr, ytr, params); fit_t = time.time() - t0
    t0 = time.time(); pte = pf(Xte); pred_t = time.time() - t0
    persist_sk(model, mname, tag, key_split, arm)
    record(key, yte, pf(Xva), pte, fit_t, pred_t)

def class_weights_safe(ytr):
    present = np.unique(ytr)
    w = compute_class_weight("balanced", classes=present, y=ytr)
    cw = {int(c): 1.0 for c in range(C)}
    cw.update({int(c): float(v) for c, v in zip(present, w)}); return cw

def train_deep(mname, cols, key_split, seed, tag, params=None, arm="default"):
    key = f"{mname}{'' if arm == 'default' else '#tuned'}|{tag}|{key_split}"
    if done(key): return
    q = {**DEFAULTS_DEEP[mname], **(params or {})}
    (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(cols, key_split)
    tf.keras.utils.set_random_seed(seed)
    m = DEEP[mname](Xtr.shape[1], q)
    m.compile(optimizer=tf.keras.optimizers.Adam(q["lr"]),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    h = m.fit(shape_for(mname, Xtr), ytr, validation_data=(shape_for(mname, Xva), yva),
              epochs=30, batch_size=q["bs"], class_weight=class_weights_safe(ytr), verbose=0,
              callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                 restore_best_weights=True)])
    fit_t = time.time() - t0
    t0 = time.time(); pte = m.predict(shape_for(mname, Xte), batch_size=1024, verbose=0)
    pred_t = time.time() - t0
    pva = m.predict(shape_for(mname, Xva), batch_size=1024, verbose=0)
    RESULTS["history"][key] = {k: [float(x) for x in v] for k, v in h.history.items()}
    persist_keras(m, mname, tag, key_split, arm)
    record(key, yte, pva, pte, fit_t, pred_t)
    tf.keras.backend.clear_session()
print("boucles pretes")


## 5. Entraînement

Quatre vagues successives :

1. **§5.1** — bras `défaut`, conditions `full` et `clean` (illustration avant audit).
2. **§5.2** — **audit** : importance par permutation + *test de transférabilité mono-feature*
   (une feature dont le pouvoir prédictif seul s'effondre entre split stratifié et split
   temporel encode la **position**, pas le **comportement**) → **liste noire**.
3. **§5.3** — bras `défaut`, condition `audited` : 5 graines + split temporel.
4. **§5.4–5.5** — recherche d'hyperparamètres à budget déclaré, puis bras `réglé`.
5. **§5.6** — autoencodeur, évalué séparément.


In [ ]:
# 5.1 — Bras defaut : conditions full et clean
COND   = {"full": F_FULL, "clean": F_CLEAN}
STRATS = [f"strat_seed{s}" for s in SEEDS]
# Grille declaree : full/clean servent l'illustration avant-audit (graine 1 + temporel) ;
# la condition auditee (§5.3) porte le tableau principal (5 graines + temporel).
ILLUS_SPLITS = ["strat_seed1", "temporal"]

for tag, cols in COND.items():
    print(f"== condition {tag} ==")
    for sk_ in ILLUS_SPLITS:
        for mname in FAST:
            train_fast(mname, cols, sk_, tag)
    for mname in DEEP:
        train_deep(mname, cols, "strat_seed1", 1, tag)
print("§5.1 termine")


In [ ]:
# 5.2 — AUDIT : importance par permutation + transferabilite mono-feature -> liste noire
from sklearn.inspection import permutation_importance

if "audit" not in RESULTS:
    (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(F_CLEAN, "strat_seed1")
    ref = LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=.1,
                         n_jobs=-1, random_state=0, verbose=-1).fit(Xtr, ytr)
    rng = np.random.RandomState(0)
    six = rng.choice(len(yte), size=min(20000, len(yte)), replace=False)
    imp = permutation_importance(ref, Xte[six], yte[six], n_repeats=5, random_state=0,
                                 n_jobs=-1, scoring="accuracy")
    order = np.argsort(-imp.importances_mean)
    top = [(F_CLEAN[i], float(imp.importances_mean[i]), float(imp.importances_std[i]))
           for i in order[:20]]
    chance = RESULTS["shortcut_probes"]["chance_majority"]
    rows, flagged = [], []
    for feat, m_, s_ in top:
        a_s, a_t = probe([feat], "strat_seed1"), probe([feat], "temporal")
        short = (a_s > 3 * chance) and (a_t < .5 * a_s)
        rows.append({"feature": feat, "importance": round(m_, 4),
                     "acc seule (stratifie)": round(a_s, 3),
                     "acc seule (temporel)": round(a_t, 3),
                     "raccourci": "OUI" if short else "-"})
        if short: flagged.append(feat)
    RESULTS["audit"] = {"perm_importance_top": top, "transfer_table": rows,
                        "blacklist": POSITIONAL + flagged, "chance": chance}
    save_results()

audit_tab = pd.DataFrame(RESULTS["audit"]["transfer_table"]); display(audit_tab)
BLACKLIST = RESULTS["audit"]["blacklist"]
F_AUDIT = [c for c in F_CLEAN if c not in BLACKLIST]
RESULTS["slice60"]["features_audited"] = F_AUDIT; save_results()
print(f"\nLISTE NOIRE ({len(BLACKLIST)}) : {BLACKLIST}")
print(f"features auditees conservees : {len(F_AUDIT)}")

# Figure 8 : diagramme de transferabilite
fig, ax = plt.subplots(figsize=(5.5, 4.6))
for r_ in RESULTS["audit"]["transfer_table"]:
    c_ = "#C44E52" if r_["raccourci"] == "OUI" else "#4C72B0"
    ax.scatter(r_["acc seule (stratifie)"], r_["acc seule (temporel)"], s=34, color=c_)
    ax.annotate(r_["feature"], (r_["acc seule (stratifie)"], r_["acc seule (temporel)"]),
                fontsize=6, xytext=(3, 3), textcoords="offset points")
lim = [0, 1]; ax.plot(lim, lim, "k--", lw=.7, label="transfert parfait")
ax.axhline(RESULTS["audit"]["chance"], color="gray", ls=":", lw=.8, label="hasard")
ax.set_xlabel("accuracy de la feature seule — split stratifie")
ax.set_ylabel("accuracy de la feature seule — split temporel")
ax.set_title("Figure 8 — transferabilite des features (rouge = raccourci)")
ax.legend(fontsize=7)
plt.savefig(FIGS / "fig8_transferability.png"); plt.savefig(FIGS / "fig8_transferability.pdf"); plt.show()

# Figure d'annexe : importance par permutation
fig, ax = plt.subplots(figsize=(6, 5))
names = [t[0] for t in RESULTS["audit"]["perm_importance_top"]][::-1]
vals  = [t[1] for t in RESULTS["audit"]["perm_importance_top"]][::-1]
errs  = [t[2] for t in RESULTS["audit"]["perm_importance_top"]][::-1]
cols_ = ["#C44E52" if n_ in BLACKLIST else "#4C72B0" for n_ in names]
ax.barh(names, vals, xerr=errs, color=cols_)
ax.set_xlabel("chute d'accuracy par permutation"); ax.set_title("Importance par permutation (LightGBM)")
plt.savefig(SUPP / "A6_permutation_importance.png"); plt.savefig(SUPP / "A6_permutation_importance.pdf"); plt.show()


In [ ]:
# 5.3 — Bras defaut : condition AUDITEE (tableau principal du papier)
for sk_ in STRATS + ["temporal"]:
    print(f"== audited / {sk_} ==")
    for mname in FAST:
        if mname == "knn" and sk_ not in ("strat_seed1", "temporal"):
            continue                                  # budget k-NN declare (predict O(n_tr x n_te))
        train_fast(mname, F_AUDIT, sk_, "audited")
    for mname in DEEP:
        train_deep(mname, F_AUDIT, sk_, int(sk_[-1]) if sk_[-1].isdigit() else 1, "audited")
print("§5.3 termine")


### 5.4 Recherche d'hyperparamètres à budget déclaré

Un benchmark comparatif doit régler chaque modèle, faute de quoi le classement mesure la
qualité des valeurs par défaut. Nous appliquons une **recherche aléatoire à budget égal
et déclaré** :

- **Sélection** : macro-F1 sur la partition de **validation** (jamais le test), condition
  `audited`, graine 1.
- **Budget** : $N$ tirages par modèle **et** un plafond de temps identique — le premier
  atteint arrête la recherche ; le nombre effectif de tirages est journalisé.
- **Coût borné** : recherche sur un sous-échantillon stratifié du train ; la configuration
  gagnante est **réentraînée sur le train complet**, 5 graines + split temporel.
- **Le tirage 0 est toujours la configuration par défaut** : le réglage ne peut donc
  jamais dégrader un modèle.
- **Provenance préservée** : les architectures BAg-IDS publiées restent évaluées dans le
  bras `défaut`. Le papier rapporte les deux bras, et leur écart répond à une question que
  les benchmarks IDS ne posent jamais : *le classement est-il un artefact du réglage ?*


In [ ]:
# 5.4a — Espaces de recherche et budget
HPO_TRIALS_FAST = 20
HPO_TRIALS_DEEP = 10
HPO_TIME_CAP_S  = 900       # plafond identique par modele
HPO_SUB_TRAIN   = 60000
HPO_SUB_VAL     = 30000

SPACE_SK = {
    "logreg":   lambda r: {"C": float(10 ** r.uniform(-3, 2)), "max_iter": 1000},
    "nb":       lambda r: {"var_smoothing": float(10 ** r.uniform(-11, -6))},
    "knn":      lambda r: {"n_neighbors": int(r.choice([3, 5, 7, 11, 15, 21])),
                           "weights": str(r.choice(["uniform", "distance"])),
                           "p": int(r.choice([1, 2]))},
    "rf":       lambda r: {"n_estimators": int(r.choice([100, 200, 300, 500])),
                           "max_depth": (None if r.rand() < .3 else int(r.choice([8, 16, 24, 32]))),
                           "min_samples_leaf": int(r.choice([1, 2, 5, 10])),
                           "max_features": str(r.choice(["sqrt", "log2"]))},
    "xgboost":  lambda r: {"n_estimators": int(r.choice([200, 300, 500, 800])),
                           "max_depth": int(r.choice([4, 6, 8, 10, 12])),
                           "learning_rate": float(10 ** r.uniform(-2, -.7)),
                           "subsample": float(r.choice([.6, .8, 1.])),
                           "colsample_bytree": float(r.choice([.6, .8, 1.])),
                           "min_child_weight": int(r.choice([1, 5, 20]))},
    "lightgbm": lambda r: {"n_estimators": int(r.choice([200, 300, 500, 800])),
                           "num_leaves": int(r.choice([31, 63, 127, 255])),
                           "learning_rate": float(10 ** r.uniform(-2, -.7)),
                           "min_child_samples": int(r.choice([5, 20, 50, 100])),
                           "subsample": float(r.choice([.6, .8, 1.])),
                           "colsample_bytree": float(r.choice([.6, .8, 1.]))}}
SPACE_DEEP = {
    "dnn": lambda r: {"h1": int(r.choice([64, 128, 256])), "h2": int(r.choice([32, 64, 128])),
                      "d1": float(r.choice([.1, .2, .3, .5])), "d2": float(r.choice([0., .1, .2, .3])),
                      "lr": float(10 ** r.uniform(-4, -2.3)), "bs": int(r.choice([128, 256, 512]))},
    "cnn": lambda r: {"f1": int(r.choice([32, 64, 128])), "f2": int(r.choice([16, 32, 64])),
                      "dense": int(r.choice([32, 64, 128])), "drop": float(r.choice([.1, .2, .3, .5])),
                      "lr": float(10 ** r.uniform(-4, -2.3)), "bs": int(r.choice([128, 256, 512]))},
    "rnn": lambda r: {"units": int(r.choice([32, 64, 128])), "dense": int(r.choice([32, 64, 128])),
                      "drop": float(r.choice([.1, .2, .3, .5])),
                      "lr": float(10 ** r.uniform(-4, -2.3)), "bs": int(r.choice([128, 256, 512]))},
    "ftt": lambda r: {"d": int(r.choice([32, 64, 96])), "heads": int(r.choice([4, 8])),
                      "blocks": int(r.choice([2, 3, 4])), "ff": int(r.choice([64, 128, 256])),
                      "drop": float(r.choice([0., .1, .2])),
                      "lr": float(10 ** r.uniform(-4, -2.7)), "bs": int(r.choice([256, 512]))}}
TUNABLE = [m for m in FAST if m != "majority"] + list(DEEP)
print(f"budget : {HPO_TRIALS_FAST} tirages (rapides) / {HPO_TRIALS_DEEP} (profonds), "
      f"plafond {HPO_TIME_CAP_S} s par modele, recherche sur <= {HPO_SUB_TRAIN:,} flux")


In [ ]:
# 5.4b — Execution de la recherche (selection sur la VALIDATION uniquement)
def _sub(X, yy, n, seed=0):
    if len(yy) <= n: return X, yy
    ix, _ = train_test_split(np.arange(len(yy)), train_size=n, random_state=seed, stratify=yy)
    return X[ix], yy[ix]
def macro_f1(y_true, pred):
    return float(f1_score(y_true, pred, labels=np.unique(y_true), average="macro", zero_division=0))

def hpo_model(mname, cols, key_split="strat_seed1"):
    (Xtr, Xva, _), (ytr, yva, _) = make_xy(cols, key_split)
    Xs, ys = _sub(Xtr, ytr, HPO_SUB_TRAIN); Xv, yv = _sub(Xva, yva, HPO_SUB_VAL)
    is_deep = mname in DEEP
    space  = (SPACE_DEEP if is_deep else SPACE_SK)[mname]
    budget = HPO_TRIALS_DEEP if is_deep else HPO_TRIALS_FAST
    r = np.random.RandomState(0); t_start = time.time(); trials = []
    cands = [{}] + [space(r) for _ in range(budget - 1)]     # tirage 0 = configuration par defaut
    for i, p in enumerate(cands):
        if time.time() - t_start > HPO_TIME_CAP_S:
            print(f"    plafond de temps atteint apres {i} tirages"); break
        try:
            if is_deep:
                q = {**DEFAULTS_DEEP[mname], **p}
                tf.keras.utils.set_random_seed(1)
                m = DEEP[mname](Xs.shape[1], q)
                m.compile(optimizer=tf.keras.optimizers.Adam(q["lr"]),
                          loss="sparse_categorical_crossentropy")
                m.fit(shape_for(mname, Xs), ys, validation_data=(shape_for(mname, Xv), yv),
                      epochs=12, batch_size=q["bs"], class_weight=class_weights_safe(ys),
                      verbose=0, callbacks=[callbacks.EarlyStopping(monitor="val_loss",
                                            patience=3, restore_best_weights=True)])
                pred = m.predict(shape_for(mname, Xv), batch_size=1024, verbose=0).argmax(1)
                tf.keras.backend.clear_session()
            else:
                _, pf = fit_sk(mname, Xs, ys, p); pred = pf(Xv).argmax(1)
            trials.append({"params": p, "val_macro_f1": macro_f1(yv, pred)})
        except Exception as e:
            trials.append({"params": p, "val_macro_f1": None, "error": str(e)[:150]})
    ok = [t for t in trials if t["val_macro_f1"] is not None]
    best = max(ok, key=lambda t: t["val_macro_f1"])
    return {"best_params": best["params"], "best_val_macro_f1": best["val_macro_f1"],
            "default_val_macro_f1": ok[0]["val_macro_f1"], "n_trials": len(trials),
            "search_time_s": round(time.time() - t_start, 1), "trials": trials}

if RUN_HPO:
    RESULTS.setdefault("hpo", {})
    for mname in TUNABLE:
        if mname in RESULTS["hpo"]: continue
        print(f"  recherche {mname} …")
        RESULTS["hpo"][mname] = h = hpo_model(mname, F_AUDIT)
        print(f"    {mname:9s} defaut {h['default_val_macro_f1']:.4f} -> regle "
              f"{h['best_val_macro_f1']:.4f}  ({h['n_trials']} tirages, {h['search_time_s']:.0f} s)")
        print(f"    parametres : {h['best_params']}")
        save_results()

if "hpo" in RESULTS:
    hpo_tab = pd.DataFrame([{"modele": m,
                             "val macro-F1 defaut": round(h["default_val_macro_f1"], 4),
                             "val macro-F1 regle": round(h["best_val_macro_f1"], 4),
                             "gain": round(h["best_val_macro_f1"] - h["default_val_macro_f1"], 4),
                             "tirages": h["n_trials"], "temps (s)": h["search_time_s"]}
                            for m, h in RESULTS["hpo"].items()]).set_index("modele")
    display(hpo_tab)
    (MODELS / "hpo_best_params.json").write_text(json.dumps(
        {m: h["best_params"] for m, h in RESULTS["hpo"].items()}, indent=1), encoding="utf-8")


In [ ]:
# 5.5 — Bras REGLE : reentrainement des configurations gagnantes sur le train complet
if RUN_HPO and "hpo" in RESULTS:
    TUNED = {m: h["best_params"] for m, h in RESULTS["hpo"].items()}
    for sk_ in STRATS + ["temporal"]:
        print(f"== audited#tuned / {sk_} ==")
        for mname in TUNABLE:
            if mname == "knn" and sk_ not in ("strat_seed1", "temporal"): continue
            if mname in DEEP:
                train_deep(mname, F_AUDIT, sk_, int(sk_[-1]) if sk_[-1].isdigit() else 1,
                           "audited", TUNED[mname], arm="tuned")
            else:
                train_fast(mname, F_AUDIT, sk_, "audited", TUNED[mname], arm="tuned")
    print("§5.5 termine")


In [ ]:
# 5.6 — Autoencodeur : entraine sur le BENIN seul, evalue en AUROC (section separee)
if "autoencoder" not in RESULTS:
    ae_res = {}
    for s in SEEDS:
        (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(F_AUDIT, f"strat_seed{s}")
        tf.keras.utils.set_random_seed(s)
        ae = build_ae(Xtr.shape[1]); ae.compile(optimizer="adam", loss="mse")
        bt, bv = Xtr[ytr == BENIGN_IDX], Xva[yva == BENIGN_IDX]
        ae.fit(bt, bt, validation_data=(bv, bv), epochs=50, batch_size=256, verbose=0,
               callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                  restore_best_weights=True)])
        err = np.mean((ae.predict(Xte, batch_size=1024, verbose=0) - Xte) ** 2, axis=1)
        per_fam = {CLASS_NAMES[i]: float(roc_auc_score(
                       yte[(yte == i) | (yte == BENIGN_IDX)] == i,
                       err[(yte == i) | (yte == BENIGN_IDX)]))
                   for i in range(C) if i != BENIGN_IDX and (yte == i).any()}
        ae_res[f"seed{s}"] = {"auroc_global": float(roc_auc_score(yte != BENIGN_IDX, err)),
                              "auroc_per_family": per_fam}
        if s == 1:
            if SAVE_MODELS: ae.save(MODELS / "ae_default_audited_strat_seed1.keras")
            np.savez_compressed(SAVE / "ae_scores_seed1.npz", err=err.astype(np.float32), y=yte)
        print(f"  AE graine {s} : AUROC {ae_res[f'seed{s}']['auroc_global']:.4f}")
        tf.keras.backend.clear_session()
    RESULTS["autoencoder"] = ae_res; save_results()

g = [v["auroc_global"] for v in RESULTS["autoencoder"].values()]
print(f"\nAUROC global : {np.mean(g):.4f} +/- {np.std(g):.4f}")
fam = pd.DataFrame({k: v["auroc_per_family"] for k, v in RESULTS["autoencoder"].items()})
display(fam.assign(moyenne=fam.mean(axis=1).round(4)).round(4))


## 6. Évaluation, figures et interprétation

Figures de l'article dans `figures/` (Fig. 1–8), figures d'annexe pour le rapport de thèse
dans `figures_annexe/` (A1–A15), fragments LaTeX dans `tables/`. Tout est exporté en
PNG **et** PDF à 300 dpi.


In [ ]:
# 6.1 — Tableau principal : macro-F1 par condition et par bras
def agg(mname, tag, metric="macro_f1"):
    v = [RESULTS["models"][f"{mname}|{tag}|strat_seed{s}"][metric]
         for s in SEEDS if f"{mname}|{tag}|strat_seed{s}" in RESULTS["models"]]
    return (float(np.mean(v)), float(np.std(v))) if v else (np.nan, np.nan)
def one(mname, tag, split, metric="macro_f1"):
    return RESULTS["models"].get(f"{mname}|{tag}|{split}", {}).get(metric, np.nan)
def pm(mu, sd): return "--" if np.isnan(mu) else f"{mu:.4f} +/- {sd:.4f}"

rows = []
for m_ in ALL_MODELS:
    rows.append({"modele": m_,
                 "full (s1)": round(one(m_, "full", "strat_seed1"), 4),
                 "clean (s1)": round(one(m_, "clean", "strat_seed1"), 4),
                 "audited defaut (5g)": pm(*agg(m_, "audited")),
                 "audited regle (5g)": pm(*agg(m_ + "#tuned", "audited")),
                 "audited defaut (temporel)": round(one(m_, "audited", "temporal"), 4),
                 "audited regle (temporel)": round(one(m_ + "#tuned", "audited", "temporal"), 4),
                 "MCC (audited s1)": round(one(m_, "audited", "strat_seed1", "mcc"), 4),
                 "FPR (audited s1)": (lambda f: round(f, 5) if f is not None else None)(
                     RESULTS["models"].get(f"{m_}|audited|strat_seed1", {}).get("binary", {}).get("fpr"))})
main_tab = pd.DataFrame(rows).set_index("modele")
display(main_tab)
RESULTS["main_table"] = main_tab.reset_index().to_dict("records"); save_results()

def esc(x): return "--" if (isinstance(x, float) and np.isnan(x)) else (
    f"{x:.4f}" if isinstance(x, float) else str(x).replace("+/-", "$\\pm$"))
(TABS / "t1_intervals.tex").write_text("\n".join(
    f"{iv}\\,s & {RESULTS['interval_stats'][iv]['total']:,} & "
    f"{RESULTS['interval_stats'][iv]['benign']:,} & "
    f"{RESULTS['interval_stats'][iv]['benign_share']*100:.1f}\\% \\\\" for iv in INTERVALS))
(TABS / "t2_main.tex").write_text("\n".join(
    " & ".join([i] + [esc(r[c]) for c in main_tab.columns]) + " \\\\"
    for i, r in main_tab.iterrows()))
if "hpo" in RESULTS:
    (TABS / "t3_hpo.tex").write_text("\n".join(
        f"{m_} & {h['n_trials']} & {h['search_time_s']:.0f} & "
        f"{h['default_val_macro_f1']:.4f} & {h['best_val_macro_f1']:.4f} \\\\"
        for m_, h in RESULTS["hpo"].items()))
print("tables ecrites :", sorted(p.name for p in TABS.glob("*.tex")))


In [ ]:
# 6.2 — Figure 2 : effet de l'audit et du split temporel sur le classement
CONDS = [("full\n(stratifie)",     lambda m: one(m, "full", "strat_seed1")),
         ("clean\n(stratifie)",    lambda m: one(m, "clean", "strat_seed1")),
         ("audited\n(stratifie)",  lambda m: agg(m, "audited")[0]),
         ("clean\n(temporel)",     lambda m: one(m, "clean", "temporal")),
         ("audited\n(temporel)",   lambda m: one(m, "audited", "temporal"))]
fig, ax = plt.subplots(figsize=(8, 4.6))
for m_ in ALL_MODELS:
    if m_ == "majority": continue
    ax.plot(range(len(CONDS)), [f(m_) for _, f in CONDS], marker="o", ms=4, lw=1.2, label=m_)
ax.set_xticks(range(len(CONDS))); ax.set_xticklabels([c for c, _ in CONDS])
ax.set_ylabel("macro-F1"); ax.set_ylim(bottom=0)
ax.set_title("Figure 2 — effet de l'audit et du protocole temporel sur le classement (60 s)")
ax.legend(fontsize=7, ncol=2, loc="lower left")
plt.savefig(FIGS / "fig2_before_after.png"); plt.savefig(FIGS / "fig2_before_after.pdf"); plt.show()
# Lecture : l'ecart entre les colonnes stratifiees et temporelles quantifie la part de
# performance imputable a la memorisation de la capture (reponse a RQ2).


In [ ]:
# 6.3 — Calibration : temperature scaling, ECE, diagrammes de fiabilite (Figure 4)
from scipy.optimize import minimize_scalar

def fit_temperature(pva, yva):
    logp = np.log(np.clip(pva, 1e-12, 1))
    def nll(T):
        q = logp / T; q -= q.max(1, keepdims=True)
        p = np.exp(q); p /= p.sum(1, keepdims=True)
        return -np.mean(np.log(np.clip(p[np.arange(len(yva)), yva], 1e-12, 1)))
    return float(minimize_scalar(nll, bounds=(.05, 10.), method="bounded").x)
def apply_T(p, T):
    q = np.log(np.clip(p, 1e-12, 1)) / T; q -= q.max(1, keepdims=True)
    e = np.exp(q); return e / e.sum(1, keepdims=True)
def ece(p, y_true, bins=ECE_BINS):
    conf, acc = p.max(1), (p.argmax(1) == y_true).astype(float)
    ed = np.linspace(0, 1, bins + 1); out = 0.
    for lo, hi in zip(ed[:-1], ed[1:]):
        m_ = (conf > lo) & (conf <= hi)
        if m_.any(): out += m_.mean() * abs(acc[m_].mean() - conf[m_].mean())
    return float(out)

MAIN = [m for m in ALL_MODELS if m != "majority"]
yva_ref, yte_ref = y[splits["strat_seed1"][1]], y[splits["strat_seed1"][2]]
if "calibration" not in RESULTS:
    cal = {}
    for m_ in MAIN:
        key = f"{m_}|audited|strat_seed1"
        if not done(key): continue
        pva, pte = load_probs(key); T = fit_temperature(pva, yva_ref)
        cal[m_] = {"T": T, "ece_before": ece(pte, yte_ref), "ece_after": ece(apply_T(pte, T), yte_ref)}
    RESULTS["calibration"] = cal; save_results()
display(pd.DataFrame(RESULTS["calibration"]).T.round(4))

nplt = len(MAIN); ncol = 5; nrow = int(np.ceil(nplt / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.7 * nrow), sharex=True, sharey=True)
for ax, m_ in zip(np.atleast_1d(axes).flat, MAIN):
    key = f"{m_}|audited|strat_seed1"
    if not done(key) or m_ not in RESULTS["calibration"]: ax.axis("off"); continue
    _, pte = load_probs(key); T = RESULTS["calibration"][m_]["T"]
    for p, lab in [(pte, "brut"), (apply_T(pte, T), "calibre")]:
        conf, pred = p.max(1), p.argmax(1); xs, ys_ = [], []
        for lo, hi in zip(np.linspace(0, 1, 16)[:-1], np.linspace(0, 1, 16)[1:]):
            m2 = (conf > lo) & (conf <= hi)
            if m2.any(): xs.append(conf[m2].mean()); ys_.append((pred[m2] == yte_ref[m2]).mean())
        ax.plot(xs, ys_, marker="o", ms=2.5, label=lab)
    ax.plot([0, 1], [0, 1], "k--", lw=.6)
    ax.set_title(f"{m_} (T={T:.2f})", fontsize=8)
for ax in np.atleast_1d(axes).flat[nplt:]: ax.axis("off")
np.atleast_1d(axes).flat[0].legend(fontsize=7)
fig.suptitle("Figure 4 — diagrammes de fiabilite (condition auditee, graine 1)")
plt.savefig(FIGS / "fig4_reliability.png"); plt.savefig(FIGS / "fig4_reliability.pdf"); plt.show()


In [ ]:
# 6.4 — Multi-intervalles 5/10/30 s (LightGBM, XGBoost, DNN — condition auditee) + Figure 3
if RUN_INTERVALS:
    RESULTS.setdefault("intervals", {})
    for iv in ["5", "10", "30"]:
        if iv in RESULTS["intervals"]: continue
        print(f"== intervalle {iv} s ==")
        d_iv, y9_iv, t_iv = load_slice(iv)
        Xiv, _, cleanc, _, _, _ = feature_sets(d_iv)
        audc = [c for c in cleanc if c not in BLACKLIST]
        y_iv = le.transform(y9_iv); idx = np.arange(len(y_iv))
        res_iv = {"n": int(len(y_iv)), "benign_share": float((y_iv == BENIGN_IDX).mean()), "runs": {}}
        stp = d_iv["StartTime"].astype(np.float64).values.reshape(-1, 1)
        for s in [1, 2, 3]:
            itr, itmp = train_test_split(idx, test_size=.4, random_state=s, stratify=y_iv)
            iva_, ite_ = train_test_split(itmp, test_size=.5, random_state=s, stratify=y_iv[itmp])
            if s == 1:
                dtc = DecisionTreeClassifier(random_state=1).fit(stp[itr], y_iv[itr])
                res_iv["probe_starttime"] = float((dtc.predict(stp[ite_]) == y_iv[ite_]).mean())
            Xa = Xiv[audc].values; sc = RobustScaler().fit(Xa[itr])
            Xtr, Xva_, Xte = [np.nan_to_num(sc.transform(Xa[i_]), nan=0., posinf=0., neginf=0.).astype(np.float32)
                              for i_ in (itr, iva_, ite_)]
            ytr_, yte_ = y_iv[itr], y_iv[ite_]
            for mname in ["lightgbm", "xgboost"]:
                t0 = time.time(); _, pf = fit_sk(mname, Xtr, ytr_); ft = time.time() - t0
                t0 = time.time(); pte = pf(Xte); pt = time.time() - t0
                res_iv["runs"][f"{mname}|seed{s}"] = evaluate(yte_, pte, ft, pt)
            tf.keras.utils.set_random_seed(s)
            m = build_dnn(Xtr.shape[1])
            m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
            t0 = time.time()
            m.fit(Xtr, ytr_, validation_data=(Xva_, y_iv[iva_]), epochs=30, batch_size=256,
                  class_weight=class_weights_safe(ytr_), verbose=0,
                  callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                     restore_best_weights=True)])
            ft = time.time() - t0
            t0 = time.time(); pte = m.predict(Xte, batch_size=1024, verbose=0); pt = time.time() - t0
            res_iv["runs"][f"dnn|seed{s}"] = evaluate(yte_, pte, ft, pt)
            tf.keras.backend.clear_session(); print(f"  graine {s} terminee")
        RESULTS["intervals"][iv] = res_iv; save_results(); del d_iv, Xiv

if RESULTS.get("intervals"):
    det = pd.DataFrame(index=CLASS_NAMES, columns=INTERVALS, dtype=float)
    for iv in INTERVALS:
        if iv == "60":
            pcf = RESULTS["models"].get("lightgbm|audited|strat_seed1", {}).get("per_class_f1", {})
        else:
            pcs = [r["per_class_f1"] for k, r in RESULTS["intervals"].get(iv, {}).get("runs", {}).items()
                   if k.startswith("lightgbm")]
            pcf = {cn: float(np.mean([p[cn] for p in pcs])) for cn in CLASS_NAMES} if pcs else {}
        for cn in CLASS_NAMES: det.loc[cn, iv] = pcf.get(cn, np.nan)
    fig, ax = plt.subplots(figsize=(5.6, 3.9))
    im = ax.imshow(det.values.astype(float), aspect="auto", vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(4)); ax.set_xticklabels([f"{iv} s" for iv in INTERVALS])
    ax.set_yticks(range(C)); ax.set_yticklabels(CLASS_NAMES)
    for i in range(C):
        for j in range(4):
            v = det.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if v < .6 else "black")
    plt.colorbar(im, label="F1 par classe (LightGBM, audite)")
    ax.set_title("Figure 3 — detectabilite par famille selon l'intervalle")
    plt.savefig(FIGS / "fig3_interval_heatmap.png"); plt.savefig(FIGS / "fig3_interval_heatmap.pdf"); plt.show()


In [ ]:
# 6.5 — Banc de cout CPU (Figure 5) : latence batch-1, debit batch-512, taille
if RUN_COST and "cost" not in RESULTS:
    cost = {}
    (Xtr, _, Xte), (ytr, _, _) = make_xy(F_AUDIT, "strat_seed1")
    n1, nb = 200, 20; x1, xb = Xte[:n1], Xte[:512]
    for mname in FAST:
        model, pf = fit_sk(mname, Xtr, ytr)
        lat = []
        for i in range(n1):
            t0 = time.perf_counter(); pf(x1[i:i+1]); lat.append(time.perf_counter() - t0)
        t0 = time.perf_counter()
        for _ in range(nb): pf(xb)
        thr = nb * 512 / (time.perf_counter() - t0)
        buf = io.BytesIO(); pickle.dump(model, buf)
        cost[mname] = {"lat_p50_ms": float(np.percentile(lat, 50) * 1e3),
                       "lat_p99_ms": float(np.percentile(lat, 99) * 1e3),
                       "throughput_512": float(thr), "size_mb": buf.getbuffer().nbytes / 1e6}
    with tf.device("/CPU:0"):
        for mname in DEEP:
            path = MODELS / f"{mname}_default_audited_strat_seed1.keras"
            if not path.exists(): continue
            m = tf.keras.models.load_model(path, compile=False, custom_objects=CUSTOM_OBJECTS)
            xin = (lambda a: a) if mname in ("dnn", "ftt") else (lambda a: a.reshape(-1, a.shape[1], 1))
            m.predict(xin(x1[:8]), verbose=0)
            lat = []
            for i in range(n1):
                t0 = time.perf_counter(); m.predict(xin(x1[i:i+1]), verbose=0)
                lat.append(time.perf_counter() - t0)
            t0 = time.perf_counter()
            for _ in range(nb): m.predict(xin(xb), verbose=0)
            cost[mname] = {"lat_p50_ms": float(np.percentile(lat, 50) * 1e3),
                           "lat_p99_ms": float(np.percentile(lat, 99) * 1e3),
                           "throughput_512": float(nb * 512 / (time.perf_counter() - t0)),
                           "size_mb": path.stat().st_size / 1e6, "params": int(m.count_params())}
    RESULTS["cost"] = cost; save_results()

if "cost" in RESULTS:
    display(pd.DataFrame(RESULTS["cost"]).T.round(3))
    (TABS / "t4_cost.tex").write_text("\n".join(
        f"{m_} & {c_['lat_p50_ms']:.2f} & {c_['lat_p99_ms']:.2f} & "
        f"{c_['throughput_512']:,.0f} & {c_['size_mb']:.2f} \\\\" for m_, c_ in RESULTS["cost"].items()))
    fig, ax = plt.subplots(figsize=(5.8, 4.2))
    for m_, c_ in RESULTS["cost"].items():
        mf1 = agg(m_, "audited")[0]
        if np.isnan(mf1): continue
        ax.scatter(c_["throughput_512"], mf1, s=32, color="#4C72B0")
        ax.annotate(m_, (c_["throughput_512"], mf1), fontsize=7, xytext=(4, 3),
                    textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("debit CPU (flux/s, batch 512, echelle log)")
    ax.set_ylabel("macro-F1 (audite, 5 graines)")
    ax.set_title("Figure 5 — compromis cout d'inference / performance")
    plt.savefig(FIGS / "fig5_cost.png"); plt.savefig(FIGS / "fig5_cost.pdf"); plt.show()


In [ ]:
# 6.6 — Tests statistiques : McNemar apparie + correction de Holm, bootstrap
from scipy.stats import chi2 as chi2dist
def mcnemar_p(pa, pb, yt):
    ca, cb = pa == yt, pb == yt
    b_, c_ = int((ca & ~cb).sum()), int((~ca & cb).sum())
    return 1.0 if b_ + c_ == 0 else float(chi2dist.sf((abs(b_ - c_) - 1) ** 2 / (b_ + c_), 1))

if "stats" not in RESULTS:
    preds = {m_: load_probs(f"{m_}|audited|strat_seed1")[1].argmax(1)
             for m_ in MAIN if done(f"{m_}|audited|strat_seed1")}
    raw = {f"{a}|{b}": mcnemar_p(preds[a], preds[b], yte_ref)
           for a, b in itertools.combinations(sorted(preds), 2)}
    order = sorted(raw, key=raw.get); mt = len(order)
    holm = {k: min(1., raw[k] * (mt - i)) for i, k in enumerate(order)}
    rng = np.random.RandomState(0); boot = {}
    for m_, pr_ in preds.items():
        vals = [f1_score(yte_ref[ix], pr_[ix], average="macro", zero_division=0)
                for ix in (rng.randint(0, len(yte_ref), len(yte_ref)) for _ in range(BOOTSTRAP_B))]
        boot[m_] = {"macro_f1_mean": float(np.mean(vals)),
                    "macro_f1_ci95": [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]}
    RESULTS["stats"] = {"mcnemar_raw": raw, "mcnemar_holm": holm, "bootstrap": boot}
    save_results()

sig = {k: v for k, v in RESULTS["stats"]["mcnemar_holm"].items() if v < .05}
print(f"paires significativement differentes (Holm < 0,05) : {len(sig)}/{len(RESULTS['stats']['mcnemar_holm'])}")
bt = RESULTS["stats"]["bootstrap"]
fig, ax = plt.subplots(figsize=(6, 3.6))
ms = sorted(bt, key=lambda m_: bt[m_]["macro_f1_mean"])
mu = [bt[m_]["macro_f1_mean"] for m_ in ms]
lo = [mu[i] - bt[m_]["macro_f1_ci95"][0] for i, m_ in enumerate(ms)]
hi = [bt[m_]["macro_f1_ci95"][1] - mu[i] for i, m_ in enumerate(ms)]
ax.errorbar(mu, range(len(ms)), xerr=[lo, hi], fmt="o", ms=4, capsize=3, color="#4C72B0")
ax.set_yticks(range(len(ms))); ax.set_yticklabels(ms)
ax.set_xlabel("macro-F1 (IC 95 % bootstrap, condition auditee)")
ax.set_title("Figure 7 — classement avec intervalles de confiance")
plt.savefig(FIGS / "fig7_ranking_ci.png"); plt.savefig(FIGS / "fig7_ranking_ci.pdf"); plt.show()


In [ ]:
# 6.7 — Figure 6 : matrice de confusion du meilleur modele
best = max((m_ for m_ in ALL_MODELS if m_ != "majority"),
           key=lambda m_: (agg(m_, "audited")[0] if not np.isnan(agg(m_, "audited")[0]) else -1))
print("meilleur modele (audite, stratifie) :", best)
pte = load_probs(f"{best}|audited|strat_seed1")[1]
cm = confusion_matrix(yte_ref, pte.argmax(1), labels=range(C), normalize="true")
fig, ax = plt.subplots(figsize=(5.8, 4.9))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(C)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(C)); ax.set_yticklabels(CLASS_NAMES, fontsize=7)
for i in range(C):
    for j in range(C):
        if cm[i, j] >= .01:
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if cm[i, j] > .5 else "black")
ax.set_xlabel("classe predite"); ax.set_ylabel("classe reelle")
ax.set_title(f"Figure 6 — matrice de confusion, {best} (audite, graine 1)")
plt.colorbar(im); plt.savefig(FIGS / "fig6_confusion.png"); plt.savefig(FIGS / "fig6_confusion.pdf"); plt.show()
print(classification_report(yte_ref, pte.argmax(1), target_names=CLASS_NAMES, digits=3, zero_division=0))


### 6.8 Figures d'annexe (rapport de thèse)

Jeu complémentaire, plus descriptif : courbes d'apprentissage, matrices de confusion de
tous les modèles, courbes ROC et précision–rappel, F1 par classe, gain du réglage,
scores de l'autoencodeur, matrice de significativité et temps d'entraînement.


In [ ]:
# 6.8a — Courbes d'apprentissage des modeles profonds (A7)
hist_keys = [f"{m_}|audited|strat_seed1" for m_ in DEEP if f"{m_}|audited|strat_seed1" in RESULTS["history"]]
if hist_keys:
    fig, axes = plt.subplots(2, len(hist_keys), figsize=(3.1 * len(hist_keys), 5.4), squeeze=False)
    for j, k in enumerate(hist_keys):
        h = RESULTS["history"][k]; nm = k.split("|")[0]
        axes[0][j].plot(h.get("accuracy", []), label="train")
        axes[0][j].plot(h.get("val_accuracy", []), label="validation")
        axes[0][j].set_title(f"{nm} — accuracy", fontsize=9); axes[0][j].set_xlabel("epoque")
        axes[1][j].plot(h.get("loss", []), label="train")
        axes[1][j].plot(h.get("val_loss", []), label="validation")
        axes[1][j].set_title(f"{nm} — perte", fontsize=9); axes[1][j].set_xlabel("epoque")
    axes[0][0].legend(fontsize=7); axes[1][0].legend(fontsize=7)
    fig.suptitle("A7 — courbes d'apprentissage (condition auditee, graine 1)")
    plt.savefig(SUPP / "A7_learning_curves.png"); plt.savefig(SUPP / "A7_learning_curves.pdf"); plt.show()


In [ ]:
# 6.8b — Matrices de confusion de TOUS les modeles (A8)
avail = [m_ for m_ in ALL_MODELS if done(f"{m_}|audited|strat_seed1")]
ncol = 4; nrow = int(np.ceil(len(avail) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.0 * nrow))
for ax, m_ in zip(np.atleast_1d(axes).flat, avail):
    p = load_probs(f"{m_}|audited|strat_seed1")[1]
    cmm = confusion_matrix(yte_ref, p.argmax(1), labels=range(C), normalize="true")
    ax.imshow(cmm, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"{m_} (mF1 {one(m_, 'audited', 'strat_seed1'):.3f})", fontsize=8)
    ax.set_xticks(range(C)); ax.set_yticks(range(C))
    ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=5)
    ax.set_yticklabels(CLASS_NAMES, fontsize=5)
for ax in np.atleast_1d(axes).flat[len(avail):]: ax.axis("off")
fig.suptitle("A8 — matrices de confusion, tous les modeles (audite, graine 1)")
plt.savefig(SUPP / "A8_confusion_all.png"); plt.savefig(SUPP / "A8_confusion_all.pdf"); plt.show()


In [ ]:
# 6.8c — Courbes ROC et precision-rappel, detection binaire (A9)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
is_att = yte_ref != BENIGN_IDX
for m_ in avail:
    if m_ == "majority": continue
    p = load_probs(f"{m_}|audited|strat_seed1")[1]; sc = 1.0 - p[:, BENIGN_IDX]
    fpr_, tpr_, _ = roc_curve(is_att, sc); pr_, rc_, _ = precision_recall_curve(is_att, sc)
    axes[0].plot(fpr_, tpr_, lw=1.2, label=f"{m_} (AUC {roc_auc_score(is_att, sc):.4f})")
    axes[1].plot(rc_, pr_, lw=1.2, label=f"{m_} (AP {average_precision_score(is_att, sc):.4f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=.6)
axes[0].set_xlabel("taux de faux positifs"); axes[0].set_ylabel("taux de vrais positifs")
axes[0].set_title("ROC — benin vs attaque"); axes[0].legend(fontsize=6, loc="lower right")
axes[1].set_xlabel("rappel"); axes[1].set_ylabel("precision")
axes[1].set_title("Precision-rappel"); axes[1].legend(fontsize=6, loc="lower left")
fig.suptitle("A9 — courbes ROC et precision-rappel (audite, graine 1)")
plt.savefig(SUPP / "A9_roc_pr.png"); plt.savefig(SUPP / "A9_roc_pr.pdf"); plt.show()


In [ ]:
# 6.8d — F1 par classe et par modele (A10) + temps d'entrainement (A11)
pcf = pd.DataFrame({m_: RESULTS["models"][f"{m_}|audited|strat_seed1"]["per_class_f1"]
                    for m_ in avail}).reindex(CLASS_NAMES)
display(pcf.round(4))
fig, ax = plt.subplots(figsize=(10, 4))
pcf.plot.bar(ax=ax, width=.85); ax.set_ylabel("F1"); ax.set_ylim(0, 1.02)
ax.set_title("A10 — F1 par classe et par modele (audite, graine 1)")
ax.legend(fontsize=6, ncol=6); plt.savefig(SUPP / "A10_per_class_f1.png")
plt.savefig(SUPP / "A10_per_class_f1.pdf"); plt.show()

ft = {m_: RESULTS["models"][f"{m_}|audited|strat_seed1"]["fit_time_s"] for m_ in avail}
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ks = sorted(ft, key=ft.get)
ax.barh(ks, [ft[k] for k in ks], color="#8172B2"); ax.set_xscale("log")
ax.set_xlabel("temps d'entrainement (s, echelle log)")
ax.set_title("A11 — cout d'entrainement par modele")
plt.savefig(SUPP / "A11_train_time.png"); plt.savefig(SUPP / "A11_train_time.pdf"); plt.show()


In [ ]:
# 6.8e — Gain du reglage (A12) et tirages de la recherche (A13)
if "hpo" in RESULTS:
    ms_ = list(RESULTS["hpo"].keys()); xs = np.arange(len(ms_)); w = .38
    d_ = [agg(m_, "audited")[0] for m_ in ms_]
    t_ = [agg(m_ + "#tuned", "audited")[0] for m_ in ms_]
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.bar(xs - w/2, d_, w, label="configuration par defaut", color="#4C72B0")
    ax.bar(xs + w/2, t_, w, label="configuration reglee", color="#DD8452")
    ax.set_xticks(xs); ax.set_xticklabels(ms_, rotation=30, ha="right")
    ax.set_ylabel("macro-F1 (audite, 5 graines)"); ax.legend(fontsize=8)
    ax.set_title("A12 — effet du reglage d'hyperparametres sur le test")
    plt.savefig(SUPP / "A12_tuning_gain.png"); plt.savefig(SUPP / "A12_tuning_gain.pdf"); plt.show()

    ncol = 4; nrow = int(np.ceil(len(ms_) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3 * ncol, 2.5 * nrow))
    for ax, m_ in zip(np.atleast_1d(axes).flat, ms_):
        v = [t["val_macro_f1"] for t in RESULTS["hpo"][m_]["trials"] if t["val_macro_f1"] is not None]
        ax.plot(range(len(v)), v, "o", ms=3, color="#4C72B0")
        ax.axhline(RESULTS["hpo"][m_]["best_val_macro_f1"], ls="--", c="#C44E52", lw=.8)
        ax.set_title(m_, fontsize=8); ax.set_xlabel("tirage", fontsize=7)
    for ax in np.atleast_1d(axes).flat[len(ms_):]: ax.axis("off")
    fig.suptitle("A13 — tirages de la recherche aleatoire (macro-F1 sur validation)")
    plt.savefig(SUPP / "A13_hpo_trials.png"); plt.savefig(SUPP / "A13_hpo_trials.pdf"); plt.show()


In [ ]:
# 6.8f — Scores de l'autoencodeur (A14) et significativite (A15)
p_ae = SAVE / "ae_scores_seed1.npz"
if p_ae.exists():
    z = np.load(p_ae); err, yv_ = z["err"], z["y"]
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    data = [np.log10(err[yv_ == i] + 1e-12) for i in range(C)]
    bp = ax.boxplot(data, labels=CLASS_NAMES, showfliers=False, patch_artist=True)
    for i, b in enumerate(bp["boxes"]):
        b.set_facecolor("#55A868" if i == BENIGN_IDX else "#C44E52"); b.set_alpha(.75)
    ax.set_ylabel("log10 de l'erreur de reconstruction")
    ax.set_title("A14 — separation des classes par l'autoencodeur (entraine sur le benin seul)")
    plt.xticks(rotation=30, ha="right")
    plt.savefig(SUPP / "A14_ae_scores.png"); plt.savefig(SUPP / "A14_ae_scores.pdf"); plt.show()

if "stats" in RESULTS:
    ms_ = sorted({k.split("|")[0] for k in RESULTS["stats"]["mcnemar_holm"]} |
                 {k.split("|")[1] for k in RESULTS["stats"]["mcnemar_holm"]})
    M = np.full((len(ms_), len(ms_)), np.nan)
    for k, v in RESULTS["stats"]["mcnemar_holm"].items():
        a, b = k.split("|"); i, j = ms_.index(a), ms_.index(b)
        M[i, j] = M[j, i] = v
    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    im = ax.imshow(M, cmap="viridis_r", vmin=0, vmax=.1)
    ax.set_xticks(range(len(ms_))); ax.set_xticklabels(ms_, rotation=90, fontsize=7)
    ax.set_yticks(range(len(ms_))); ax.set_yticklabels(ms_, fontsize=7)
    plt.colorbar(im, label="p (McNemar, correction de Holm)")
    ax.set_title("A15 — significativite des differences entre modeles")
    plt.savefig(SUPP / "A15_mcnemar.png"); plt.savefig(SUPP / "A15_mcnemar.pdf"); plt.show()


## 7. Sauvegarde des modèles et export

Le bundle contient tout le nécessaire pour rejouer une inférence sans réentraîner :
modèles de référence des deux bras, autoencodeur, paramètres du scaler, listes de
features, liste noire, encodage des classes, indices des splits gelés et manifeste.


In [ ]:
# 7.1 — Bundle de deploiement (scaler, features, classes, splits, manifeste)
if SAVE_MODELS:
    _, _, sc_audit = make_xy(F_AUDIT, "strat_seed1", return_scaler=True)
    _, _, sc_clean = make_xy(F_CLEAN, "strat_seed1", return_scaler=True)
    (MODELS / "preprocessing.json").write_text(json.dumps({
        "scaler": "RobustScaler ajuste sur le train de strat_seed1",
        "audited": {"features": F_AUDIT, "center_": sc_audit.center_.tolist(),
                    "scale_": sc_audit.scale_.tolist()},
        "clean":   {"features": F_CLEAN, "center_": sc_clean.center_.tolist(),
                    "scale_": sc_clean.scale_.tolist()},
        "class_names": CLASS_NAMES, "benign_index": BENIGN_IDX,
        "blacklist": BLACKLIST, "identifiers_excluded": IDENTIFIERS,
        "positional": POSITIONAL}, indent=1), encoding="utf-8")
    shutil.copy(SPLITS_PATH, MODELS / "frozen_splits_60s.npz")
    (MODELS / "MANIFEST.json").write_text(json.dumps({
        "paper": "A Leakage-Audited Benchmark of Deep and Ensemble Detectors on the GeNIS 2025 Corpus",
        "corpus": "GeNIS 2025, 2-flows, intervalle 60 s, 9 classes (doi:10.5281/zenodo.14919237)",
        "reference_config": "condition auditee, split strat_seed1",
        "arms": ["default", "tuned"],
        "files": sorted(p.name for p in MODELS.iterdir()),
        "calibration_temperatures": {k: v["T"] for k, v in RESULTS.get("calibration", {}).items()},
        "created": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=1), encoding="utf-8")
    tot = sum(p.stat().st_size for p in MODELS.iterdir()) / 1e6
    print(f"{len(list(MODELS.iterdir()))} fichiers dans models/ ({tot:.1f} Mo)")
    for p in sorted(MODELS.iterdir()): print("  -", p.name)


In [ ]:
# 7.2 — Archives et telechargement
save_results()
exp = pathlib.Path("/content/export"); shutil.rmtree(exp, ignore_errors=True); exp.mkdir()
shutil.copytree(FIGS, exp / "figures"); shutil.copytree(SUPP, exp / "figures_annexe")
shutil.copytree(TABS, exp / "tables"); shutil.copy(RES_PATH, exp / "article1_results.json")
shutil.make_archive("/content/article1_figures", "zip", exp)
print(f"figures + tables : {os.path.getsize('/content/article1_figures.zip')/1e6:.1f} Mo")

if SAVE_MODELS:
    shutil.make_archive(str(SAVE / "article1_models"), "zip", MODELS)
    mb = os.path.getsize(SAVE / "article1_models.zip") / 1e6
    print(f"modeles : {mb:.1f} Mo -> {SAVE/'article1_models.zip'} (conserve sur Drive)")

print("\n" + "=" * 72)
print("CHIFFRES CLES")
print("=" * 72)
pr = RESULTS["shortcut_probes"]
print(f"corpus 60 s          : {RESULTS['slice60']['n']:,} flux | {C} classes | "
      f"benin {RESULTS['slice60']['benign_share']:.1%}")
print(f"sonde StartTime seul : stratifie {pr['starttime_only']['strat_mean']:.4f} | "
      f"temporel {pr['starttime_only']['temporal']:.4f} | hasard {pr['chance_majority']:.4f}")
print(f"liste noire ({len(BLACKLIST)})     : {BLACKLIST}")
print(f"features conservees  : {len(F_AUDIT)} / {len(F_FULL)}")
mu, sd = agg(best, "audited")
print(f"meilleur modele      : {best} — macro-F1 {mu:.4f} +/- {sd:.4f} (stratifie) | "
      f"{one(best, 'audited', 'temporal'):.4f} (temporel)")
g = [v["auroc_global"] for v in RESULTS["autoencoder"].values()]
print(f"autoencodeur         : AUROC {np.mean(g):.4f} +/- {np.std(g):.4f}")
print("=" * 72)

from google.colab import files
files.download("/content/article1_figures.zip")
files.download(str(RES_PATH))
print("\nTermine. Renvoyer article1_results.json et article1_figures.zip.")
print(f"Le bundle de modeles reste sur Drive : {SAVE/'article1_models.zip'}")


---
## Récapitulatif des livrables

**`figures/` — article (300 dpi, PNG + PDF)**
| Fig. | Contenu |
|---|---|
| 1 | Chronologie des fenêtres d'attaque |
| 2 | Classement avant/après audit et protocole temporel |
| 3 | Détectabilité par famille × intervalle |
| 4 | Diagrammes de fiabilité (calibration) |
| 5 | Coût d'inférence vs performance |
| 6 | Matrice de confusion du meilleur modèle |
| 7 | Classement avec intervalles de confiance bootstrap |
| 8 | Transférabilité des features (identification des raccourcis) |

**`figures_annexe/` — rapport de thèse** : A1 intervalles · A2 distribution des classes ·
A3 corrélation des features · A4 projection PCA · A5 sondes de raccourci ·
A6 importance par permutation · A7 courbes d'apprentissage · A8 matrices de confusion de
tous les modèles · A9 ROC et précision–rappel · A10 F1 par classe · A11 temps
d'entraînement · A12 gain du réglage · A13 tirages de la recherche · A14 scores de
l'autoencodeur · A15 significativité statistique.

**`tables/`** : T1 intervalles · T2 tableau principal · T3 budget de réglage · T4 coût.

**`models/`** : modèles de référence des deux bras (`.joblib` / `.keras`), autoencodeur,
`preprocessing.json` (scaler, features, classes, liste noire), `frozen_splits_60s.npz`,
`hpo_best_params.json`, `MANIFEST.json`.

## En cas de déconnexion

Relancer *Exécution → Tout exécuter* : le notebook affiche `runs deja calcules et
reutilises : N` et saute tout ce qui est déjà fait.

## Points de vigilance à me signaler

- Sonde `StartTime` inférieure à 0,95 en stratifié → le raccourci est plus faible qu'attendu.
- Un modèle à macro-F1 ≥ 0,999 en condition auditée **et** en temporel → chercher un
  identifiant résiduel.
- Une classe absente d'une partition du split temporel → l'invariant §3.3 doit afficher `OK`.
